In [275]:
from datetime import datetime

from transformers.utils import replace_variables_in_comments

s = '2026-03-18 15:57:19.062'
s = datetime.strptime(s, '%Y-%m-%d %H:%M:%S.%f').strftime('%Y-%m-%dT%H:%M:%S.%f')[:-3]
print(s)

2026-03-18T15:57:19.062


In [276]:
import yaml
from paths import VARIABLE_CONFIG_PATH

with open(VARIABLE_CONFIG_PATH, 'r', encoding='utf-8') as f:
    variable_mapping = yaml.safe_load(f)

In [344]:
import sqlglot


# 1. Đọc nội dung tệp SQL
with open(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\logic_cleaner\before.sql", 'r') as file:
    sql_content = file.read()

original_ast = sqlglot.parse(sql_content, read='hive')
original_ast

[Select(
   expressions=[
     Alias(
       this=DPipe(
         this=Literal(this='MHBOS_', is_string=True),
         expression=Column(
           this=Identifier(this=CLIENT_NO, quoted=False),
           table=Identifier(this=T1, quoted=False)),
         safe=True),
       alias=Identifier(this=OWNER_ID, quoted=False),
       _comments=[
         None]),
     Alias(
       this=Literal(this='ACCOUNT', is_string=True),
       alias=Identifier(this=CONTACT_OWNER_TYPE, quoted=False),
       _comments=[
         None]),
     Alias(
       this=Literal(this='MOBILE', is_string=True),
       alias=Identifier(this=CONTACT_TYPE, quoted=False),
       _comments=[
         None]),
     Alias(
       this=Column(
         this=Identifier(this=MOBILE_NO, quoted=False),
         table=Identifier(this=T1, quoted=False)),
       alias=Identifier(this=CONTACT_VALUE, quoted=False),
       _comments=[
         None]),
     Alias(
       this=Null(),
       alias=Identifier(this=CONTACT_NAME, quoted=

In [347]:
import sqlglot
import sqlglot.expressions as exp

def replace_partition_columns(stmt: exp.Expression) -> exp.Expression:
    """
    Quét và biến đổi các cột etl_dt, part_id thành DATE_FORMAT({alias}.dl_record_updated_date, 'yyyyMMdd')
    chỉ khi chúng nằm trong ngữ cảnh điều kiện truy vấn (WHERE, JOIN ON, HAVING).
    """

    def is_in_condition(node: exp.Expression) -> bool:
        """
        Hàm helper đi ngược từ Node lên gốc để xem nó có thuộc nhánh điều kiện không.
        """
        current = node
        while current.parent:
            parent = current.parent

            # Bẫy DDL: Nếu đụng phải DDL thì chặn đứng ngay lập tức
            if isinstance(parent, (exp.Create, exp.Drop, exp.Alter)):
                return False

            # Luật 1: Node đang đi lên từ nhánh '.this' của mệnh đề WHERE hoặc HAVING
            if isinstance(parent, (exp.Where, exp.Having)) and parent.this is current:
                return True

            # Luật 2: Node đang đi lên từ nhánh '.args["on"]' của mệnh đề JOIN
            if isinstance(parent, exp.Join) and parent.args.get("on") is current:
                return True

            current = parent

        return False

    def transformer(node):
        # Chỉ can thiệp nếu là Cột và có tên mục tiêu
        if isinstance(node, exp.Column) and node.name.lower() in ('etl_dt', 'part_id'):

            if is_in_condition(node):
                # 1. Lấy bí danh (alias) của bảng nếu có
                table_alias = node.args.get("table")

                # 2. Tạo cột mới: {alias}.dl_record_updated_date
                new_col = exp.column("dl_record_updated_date", table=table_alias)

                # 3. Bọc trong hàm DATE_FORMAT
                # Sử dụng exp.Anonymous để buộc sqlglot xuất ra chính xác chuỗi "DATE_FORMAT"
                # mà không bị phiên dịch (transpile) sang các hàm đặc thù của Dialect khác.
                new_func = exp.Anonymous(
                    this="DATE_FORMAT",
                    expressions=[new_col, exp.Literal.string("yyyyMMdd")]
                )
                return new_func

        return node

    # Trả về AST mới sau khi đã transform bottom-up
    return stmt.transform(transformer)


# --- THỬ NGHIỆM ---
if __name__ == "__main__":
    sql_before = """
    SELECT 'MHBOS_' || T1.CLIENT_NO AS OWNER_ID, T1.ETL_DT
      FROM db.T_MHBOS_M_CLIENT AS T1
      JOIN db.T_MHBOS_M_CLIENT_EXT AS T2
        ON T2.ETL_DT = '${batch_date}'
        AND T2.part_id = '${last_date}'
        AND T1.ETL_DT  = T2.PART_ID
     WHERE T1.ETL_DT = '${last_date}'
       AND T1.part_id = '${batch_date}'
       AND T1.test_date = '${batch_date}'
       AND T1.ETL_DT IN (SELECT DISTINCT PART_ID FROM db.temp_table)
    """

    parsed_ast = sqlglot.parse_one(sql_before, read="hive")
    transformed_ast = replace_partition_columns(parsed_ast)

    print(transformed_ast.sql(dialect="hive", pretty=True))

SELECT
  'MHBOS_' || T1.CLIENT_NO AS OWNER_ID,
  T1.ETL_DT
FROM db.T_MHBOS_M_CLIENT AS T1
JOIN db.T_MHBOS_M_CLIENT_EXT AS T2
  ON DATE_FORMAT(T2.dl_record_updated_date, 'yyyyMMdd') = '${batch_date}'
  AND DATE_FORMAT(T2.dl_record_updated_date, 'yyyyMMdd') = '${last_date}'
  AND DATE_FORMAT(T1.dl_record_updated_date, 'yyyyMMdd') = DATE_FORMAT(T2.dl_record_updated_date, 'yyyyMMdd')
WHERE
  DATE_FORMAT(T1.dl_record_updated_date, 'yyyyMMdd') = '${last_date}'
  AND DATE_FORMAT(T1.dl_record_updated_date, 'yyyyMMdd') = '${batch_date}'
  AND T1.test_date = '${batch_date}'
  AND DATE_FORMAT(T1.dl_record_updated_date, 'yyyyMMdd') IN (
    SELECT DISTINCT
      DATE_FORMAT(dl_record_updated_date, 'yyyyMMdd')
    FROM db.temp_table
  )


In [342]:
from transformers.utils import replace_table_identifier, strip_partition_clauses

# from src.transformers.utils import replace_table_identifier

ast_bien_doi = replace_table_identifier(
    node=original_ast[0],
    old_schema="${com_schema}",
    old_table="t_mhbos_m_client",
    new_schema="${com_schema}",
    new_table="temp_t_mhbos_m_client_consolidated",
    dialect="hive" # Hoặc spark
)

ast_bien_doi = strip_partition_clauses(ast_bien_doi)

# In ra kết quả
print(ast_bien_doi.sql(dialect="hive", pretty=True))

SELECT
  'MHBOS_' || T1.CLIENT_NO AS OWNER_ID, /* None */
  'ACCOUNT' AS CONTACT_OWNER_TYPE, /* None */
  'MOBILE' AS CONTACT_TYPE, /* None */
  T1.MOBILE_NO AS CONTACT_VALUE, /* None */
  NULL AS CONTACT_NAME, /* None */
  T1.DATE_CREATED AS CONTACT_CREATE_DATE, /* None */
  T1.DATE_CHANGE AS CONTACT_UPDATE_DATE, /* None */
  'EB' AS LINE_OF_BUSINESS, /* None */
  'MHBOS' AS SOURCE_NAME, /* None */
  T1.CLIENT_NO AS SOURCE_RECORD_ID, /* None */
  1 AS SEQUENCE_NO /* None */
FROM ${com_schema}.temp_t_mhbos_m_client_consolidated AS T1 /* None */
JOIN ${com_schema}.T_MHBOS_M_CLIENT_EXT AS T2 /* None */
  ON DATE_FORMAT(T2.dl_record_updated_date, 'yyyyMMdd') = '{batch_date}'
  AND DATE_FORMAT(T2.dl_record_updated_date, 'yyyyMMdd') = '{last_date}'
  AND DATE_FORMAT(T1.dl_record_updated_date, 'yyyyMMdd') = DATE_FORMAT(T2.dl_record_updated_date, 'yyyyMMdd')
WHERE
  DATE_FORMAT(T1.dl_record_updated_date, 'yyyyMMdd') = '{last_date}'
  AND DATE_FORMAT(T1.dl_record_updated_date, 'yyyyMMdd') = '{

In [279]:
replace_variabled = replace_variables_in_comments(original_ast[0], variable_mapping)


with open(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\logic_cleaner\function_test\hilarious_comment.sql", "w") as f:
    f.write(replace_variabled.sql(dialect='hive', pretty=True) + ";")

In [280]:
from migration.com.key_detector import KeyDetectorV2
from utils.file_utils import parse_file_name
from migration.com.decomposer import ComSqlDecomposer
from paths import *
from src.utils.source_rule_loader import load_all_source_rules

# input_file = PROJECT_ROOT / "docs" / "datalake_old" /"dml" / "com_r_k2_cif_alias.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_t_m21_a_customer.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_t_mhbos_m_client.sql"
input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_contact.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_risk_profile.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
file_name = os.path.basename(input_file).replace('.sql', '')
layer, sub_layer, source_name, base_table = parse_file_name(input_file)
output_root = PROJECT_ROOT / "output" / "migration"

all_source_rules = load_all_source_rules()
source_rules = load_all_source_rules()[source_name] if source_name in all_source_rules else all_source_rules['default']

print(f"File gốc tại: {input_file}")

# def run_migration_pipeline():
# ==========================================
# BƯỚC 1: BÓC TÁCH SQL (DECOMPOSER)
# ==========================================
decomposer = ComSqlDecomposer(source_rules)
decomposed_script = decomposer.decompose(input_file)

# Ghi file sub-SQL ra ổ đĩa
KeyDetectorV2().detect(decomposed_script, source_rules)


ImportError: cannot import name 'KeyDetectorV2' from 'migration.key_detector' (C:\Users\dungp\projects\hql_spark_bridge\src\migration\key_detector.py)

In [333]:

from typing import Optional, Dict, Set, List
import sqlglot
import sqlglot.expressions as exp

def get_source_tables(stmt: exp.Expression) -> list[str]:
    """
    Trích xuất tất cả các bảng nguồn thực sự từ một câu lệnh SQL.
    Bỏ qua bảng đích (INSERT/CREATE) và bỏ qua các CTE.
    """
    # 1. Thu thập tên các CTE để loại trừ (ví dụ: temp_dim_account_contact_m21_row_num)
    cte_names = {cte.alias.upper() for cte in stmt.find_all(exp.CTE)}

    # 2. Thu thập "Node ID" của các bảng đích (INSERT INTO, CREATE TABLE)
    # Dùng id() để chỉ loại bỏ đúng cái node đó, không loại bỏ nhầm nếu bảng bị gọi lại
    mutated_nodes = set()
    for node in stmt.find_all((exp.Insert, exp.Create, exp.Drop, exp.TruncateTable)):
        if isinstance(node.this, exp.Schema):
            mutated_nodes.add(id(node.this.this))
        if isinstance(node.this, exp.Table):
            mutated_nodes.add(id(node.this))

    source_tables = set()

    # 3. Quét mọi node Table trong toàn bộ cây AST (bao gồm cả trong CTE, UNION)
    for table in stmt.find_all(exp.Table):
        # Bỏ qua nếu node này chính là bảng đích đang bị INSERT/CREATE
        if id(table) in mutated_nodes:
            continue

        name = table.name.upper()

        # Bỏ qua nếu bảng đang được gọi thực chất chỉ là một CTE (bảng ảo)
        if name in cte_names:
            continue

        source_tables.add(name)

    return list(source_tables)

def _strategy_partition_key(stmt: exp.Expression) -> Optional[str]:
    """
    Chiến thuật 1 (Mạnh nhất): Tìm khai báo PARTITION(SOURCE_KEY = 'XYZ')
    """
    for partition in stmt.find_all(exp.Partition):
        for eq in partition.find_all(exp.EQ):
            if isinstance(eq.left, exp.Column) and eq.left.name.upper() in ('SOURCE_KEY', 'SOURCE_NAME'):
                if isinstance(eq.right, exp.Literal):
                    return eq.right.name.upper()
    return None


def _strategy_source_key_assignment(stmt: exp.Expression) -> Optional[str]:
    """
    Chiến thuật 2: Tìm khai báo 'XYZ' AS SOURCE_NAME.
    Sqlglot find_all() tự động đệ quy xuyên qua CTEs, UNIONs, và Subqueries.
    """
    detected_sources = set()

    # Quét toàn bộ AST để tìm mọi Alias
    for alias in stmt.find_all(exp.Alias):
        if alias.alias.upper() == 'SOURCE_KEY':
            if isinstance(alias.this, exp.Literal):
                detected_sources.add(alias.this.name.upper())

    if len(detected_sources) == 1:
        return list(detected_sources)[0]
    elif len(detected_sources) > 1:
        # Nếu UNION 2 nguồn khác nhau (MHBOS và GUAVA), đây là bước gom data
        return "FINAL_CONSOLIDATION"

    return None

def _strategy_source_name_assignment(stmt: exp.Expression) -> Optional[str]:
    """
    Chiến thuật 2: Tìm khai báo 'XYZ' AS SOURCE_NAME.
    Sqlglot find_all() tự động đệ quy xuyên qua CTEs, UNIONs, và Subqueries.
    """
    detected_sources = set()

    # Quét toàn bộ AST để tìm mọi Alias
    for alias in stmt.find_all(exp.Alias):
        if alias.alias.upper() == 'SOURCE_NAME':
            if isinstance(alias.this, exp.Literal):
                detected_sources.add(alias.this.name.upper())

    if len(detected_sources) == 1:
        return list(detected_sources)[0]
    elif len(detected_sources) > 1:
        # Nếu UNION 2 nguồn khác nhau (MHBOS và GUAVA), đây là bước gom data
        return "FINAL_CONSOLIDATION"

    return None

def _strategy_registry_fallback(stmt: exp.Expression, registry: Dict[str, Set[str]]) -> Optional[str]:
    """
    Chiến thuật "Desperate Attempt":
    Đối chiếu các bảng nguồn thực sự của câu lệnh với Registry tự điển.
    """
    # Bước 1: Lấy danh sách bảng nguồn thực sự (bỏ qua CTE, bỏ qua Target Table)
    source_tables: List[str] = get_source_tables(stmt)

    if not source_tables:
        return None

    matched_sources = set()

    # Bước 2: Quét đối chiếu với Registry
    for table in source_tables:
        table_name_upper = table.upper()

        for source_name, dependent_tables in registry.items():
            # Đảm bảo các bảng trong registry cũng được viết hoa để so sánh chuẩn
            dependent_tables_upper = {t.upper() for t in dependent_tables}

            if table_name_upper in dependent_tables_upper:
                matched_sources.add(source_name)

    # Bước 3: Đánh giá kết quả
    if len(matched_sources) == 1:
        return list(matched_sources)[0]
    elif len(matched_sources) > 1:
        # Lệnh này dùng các bảng thuộc nhiều nguồn khác nhau -> Đây là bước gom (Consolidation)
        return "FINAL_CONSOLIDATION"

    return None

def _strategy_table_prefix(stmt: exp.Expression, target_table: str) -> Optional[str]:
    """
    Chiến thuật 3 (Fallback): Bắt nguồn dựa trên tiền tố của bảng trong FROM/JOIN.
    Ví dụ: T_RAK_CUSTOMER -> RAK
    """
    source_candidates = set()

    for t in stmt.find_all(exp.Table):
        name = t.name.upper()
        # Bỏ qua bảng đích nếu nó bị gọi lại trong FROM
        if name == target_table:
            continue

        parts = name.split('_')
        if len(parts) >= 2 and parts[0] in ('T', 'M', 'R'):
            source_candidates.add(parts[1])

    if len(source_candidates) == 1:
        return list(source_candidates)[0]
    elif len(source_candidates) > 1:
        return "FINAL_CONSOLIDATION"

    return None



def _strategy_mutation_query(stmt: exp.Expression, registry: Dict[str, Set[str]]):

    matched_sources = set()

    if isinstance(stmt, (exp.Create, exp.Drop, exp.TruncateTable)):
        for t in stmt.find_all(exp.Table):
            table = t.name
            table_name_upper = table.upper()

            for source_name, dependent_tables in registry.items():
                # Đảm bảo các bảng trong registry cũng được viết hoa để so sánh chuẩn
                dependent_tables_upper = {t.upper() for t in dependent_tables}

                if table_name_upper in dependent_tables_upper:
                    matched_sources.add(source_name)

    # Bước 3: Đánh giá kết quả
    if len(matched_sources) >= 1:
        return list(matched_sources)[0]

def analyze_source_id(stmt: exp.Expression, target_table: str, registry: dict) -> str:
    """
    Hàm tổng điều phối. Chạy tuần tự các chiến thuật theo độ ưu tiên.
    """
    # Bước 1: Lọc DDL khởi tạo (COMMON INIT)
    # if isinstance(stmt, (exp.Create, exp.Drop, exp.TruncateTable)):
    #     for t in stmt.find_all(exp.Table):
    #         if t.name.upper().startswith('TEMP_'):
    #             return "COMMON_INIT"

    # Bước 2: Danh sách chiến thuật và Tham số (Lazy Evaluation)
    # Định nghĩa bằng lambda để chỉ chạy khi chiến thuật trước đó thất bại (tiết kiệm Performance)
    strategies = {
        "PARTITION_KEY": lambda: _strategy_partition_key(stmt),
        "SOURCE_KEY_ASSIGNMENT": lambda: _strategy_source_key_assignment(stmt),
        "SOURCE_NAME_ASSIGNMENT": lambda: _strategy_source_name_assignment(stmt),
        "TABLE_PREFIX":  lambda: _strategy_table_prefix(stmt, target_table),
        "REGISTRY": lambda : _strategy_registry_fallback(stmt, registry),
        "MUTATION_QUERY": lambda: _strategy_mutation_query(stmt, registry),
    }

    # Bước 3: Đánh giá theo Priority
    for strategy_name, strategy_func in strategies.items():
        result = strategy_func()
        if result is not None:
            # Bạn có thể bật logging ở đây để xem script nào được bóc bởi chiến thuật nào
            print(f"Matched {result} via {strategy_name}")
            return result

    # Nếu tất cả các chiến thuật đều thất bại
    return "UNKNOWN_SOURCE"


In [336]:

{'TOMS': {'TEMP_DIM_ACCOUNT_CONTACT',
  'TEMP_DIM_ACCOUNT_CONTACT_TOMS_ECORPORATE',
  'TEMP_DIM_TRADER_CONTACT'},
 'MHBOS': {'TEMP_DIM_ACCOUNT_CONTACT',
  'TEMP_DIM_BRANCH_CONTACT',
  'TEMP_DIM_TRADER_CONTACT'},
 'GUAVA': {'TEMP_DIM_ACCOUNT_CONTACT', 'TEMP_DIM_ACCOUNT_CONTACT_GUAVA'},
 'M21': {'TEMP_DIM_ACCOUNT_CONTACT',
  'TEMP_DIM_ACCOUNT_CONTACT_M21',
  'TEMP_DIM_TRADER_CONTACT'},
 'KDI': {'TEMP_DIM_ACCOUNT_CONTACT'},
 'RAK': {'TEMP_DIM_ACCOUNT_CONTACT'},
 'SBL': {'TEMP_DIM_ACCOUNT_CONTACT'},
 'LMS': {'TEMP_DIM_ACCOUNT_CONTACT'}}


# test_file = Path(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\testing\dim_contact_lms_test.sql")
# test_file = Path(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\testing\dim_contact_toms_test.sql")
# test_file = Path(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\testing\dim_contact_m21_test.sql")
# test_file = Path(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\testing\dim_contact_m21_create.sql")
test_file = Path(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\testing\dim_contact_create.sql")
# test_file = Path(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\testing\dim_contact_test.sql")

statements = sqlglot.parse(test_file.read_text(encoding="utf-8"), read="hive", error_level=sqlglot.ErrorLevel.WARN)

# _determine_source_recursive(statements[0], 'dim_contact', test_registry)
# {
#     "source_key_strategy": _analyze_statement_source_key_strategy(statements[0], 'dim_contact'),
#     "source_name_strategy": _analyze_statement_source_name_strategy(statements[0], 'dim_contact'),
#     "select_table_strategy": _analyze_statement_select_table_strategy(statements[0], 'dim_contact')
# }

analyze_source_id(statements[0], 'dim_contact', test_registry)

Matched LMS via MUTATION_QUERY


'LMS'

In [ ]:
test = statements[0].find(exp.With).expressions[1].this

isinstance(test, exp.Union)

_find_literal_assignment(test.expression, "SOURCE_NAME")

In [302]:
get_source_tables(statements[0])


for node in statements[0].find_all((exp.Insert, exp.Create, exp.Drop, exp.TruncateTable)):
    if isinstance(node.this, exp.Schema):
        print(node.this.this.name)
    if isinstance(node.this, exp.Table):
        print(table.name)

TEMP_DIM_ACCOUNT_CONTACT


In [297]:
statements[0].this.this.name

'TEMP_DIM_ACCOUNT_CONTACT'

In [ ]:
for table in statements[0].expression.find_all(exp.Table):
    print(table.name)

In [ ]:
schema = KeyDetectorV2()._get_schema(decomposed_script, source_rules)
schema

In [ ]:
from src.migration.ddl_resolver import DdlResolver
from src.migration.schema_extractor import SchemaExtractor

ddl_resolver = DdlResolver(source_rules = source_rules)
ddl_path = ddl_resolver.resolve_ddl_path(input_file)

schema_extractor = SchemaExtractor(source_rules)
columns = schema_extractor.extract(ddl_path)

[column['name'] for column in columns]